# Explorar dados raspados
Base atual (`eventos.db`, SQLite). Rode as células e mexa à vontade.

In [1]:
import sqlite3, pandas as pd
from pathlib import Path
_p = Path.cwd()
DB = next(d/"eventos.db" for d in [_p, *_p.parents] if (d/"eventos.db").exists())
con = sqlite3.connect(DB)
q = lambda sql, *a: pd.read_sql_query(sql, con, params=a)
pd.set_option("display.max_colwidth", 70)
q("SELECT fonte, COUNT(*) n FROM eventos GROUP BY fonte ORDER BY n DESC")

,fonte,n
0,sympla,228
1,shotgun,15
2,ingresse,4


In [2]:
# todos os eventos, por data
q("SELECT fonte, start_date, nome, local_nome, cidade, url FROM eventos ORDER BY start_date")

,fonte,start_date,nome,local_nome,cidade,url
0,sympla,2025-11-14T14:00:00+00:00,Conecte-se com a Melhor Banda Larga Residencial em Brasília,Age Fibra,Brasília,https://www.sympla.com.br/evento/conecte-se-com-a-melhor-banda-lar...
1,sympla,2026-03-20T18:20:00+00:00,INVASÃO ALIEN (BRASÍLIA 02),JK Shopping,Brasília,https://www.sympla.com.br/evento/invasao-alien-brasilia-02/3355427
2,sympla,2026-06-05T16:00:00+00:00,"O Que Fazer em Brasília? Sextou com Drinks, Sinuca, Totó e Gastron...",Villa Copacabana • Happy Hour Brasília,Brasília,https://www.sympla.com.br/evento/o-que-fazer-em-brasilia-sextou-co...
3,sympla,2026-06-06T13:00:00+00:00,"O Que Fazer Domingo em Brasília? Samba, Sinuca, Totó e Petiscos no...",Villa Copacabana • Happy Hour Brasília,Brasília,https://www.sympla.com.br/evento/o-que-fazer-domingo-em-brasilia-s...
4,sympla,2026-06-06T16:00:00+00:00,"Onde Almoçar em Brasília? Sábado da Costela, Sinuca e Diversão no ...",Villa Copacabana • Happy Hour Brasília,Brasília,https://www.sympla.com.br/evento/onde-almocar-em-brasilia-sabado-d...
...,...,...,...,...,...,...
242,sympla,2026-10-29T11:00:00+00:00,International Conference of Nanoscience and Nanobiotecnology 2026,Instituto de Ciências Biológicas - IB,Brasília,https://www.sympla.com.br/evento/international-conference-of-nanos...
243,sympla,2026-11-14T21:00:00+00:00,Blood fire death festival 11° edição 2026,Galpazinho setor central gama df,Brasília,https://www.sympla.com.br/evento/blood-fire-death-festival-11deg-e...
244,sympla,2026-11-19T03:45:00+00:00,Destravando sua Timidez,Cyrela Vega Luxury Design,Brasília,https://www.sympla.com.br/evento/destravando-sua-timidez/3440486
245,sympla,2026-11-28T00:00:00+00:00,"Nexo: Funk, rap e house em Brasília",Local a definir,Brasília,https://www.sympla.com.br/evento/nexo-funk-rap-e-house-em-brasilia...


In [3]:
# busca textual (mesmo índice FTS que o agente usaria)
def buscar(termo):
    return q("SELECT fonte, start_date, nome, local_nome, url FROM eventos "
             "WHERE rowid IN (SELECT rowid FROM eventos_fts WHERE eventos_fts MATCH ?) "
             "ORDER BY start_date", termo)
buscar("pagode")

,fonte,start_date,nome,local_nome,url
0,sympla,2026-07-05T15:00:00+00:00,Varanda da Copa | Oitavas de Final + Pagode & Feijoada,Contexto Bar e Restaurante,https://www.sympla.com.br/evento/varanda-da-copa-oitavas-de-final-...
1,sympla,2026-07-05T15:00:00+00:00,Varanda da Copa | Brasil vs Noruega + Pagode + Feijoada | ADA no C...,Contexto Bar,https://www.sympla.com.br/evento/varanda-da-copa-brasil-vs-noruega...
2,sympla,2026-07-05T15:00:00+00:00,05.07 - TNP - Pagode do Jorgin,Bahrein Comida de Boteco,https://www.sympla.com.br/evento/05-07-tnp-pagode-do-jorgin/3488889
3,sympla,2026-07-05T17:00:00+00:00,Brasil + pagode,Vera cruz gastrobar,https://www.sympla.com.br/evento/brasil-pagode/3485657
4,sympla,2026-07-05T17:00:00+00:00,Jogo do Brasil + Pagode com Tá na Medida - Domingo 05.07,Brazolia Cozinha E Bar,https://www.sympla.com.br/evento/jogo-do-brasil-pagode-com-ta-na-m...
5,sympla,2026-07-05T18:00:00+00:00,05.07| Brasil nas Oitavas com Doze por Oito|Uma mesa e um pagode,Ordinário Bar e Música,https://www.sympla.com.br/evento/05-07-brasil-nas-oitavas-com-doze...
6,sympla,2026-07-19T15:00:00+00:00,SANTO DE CASA FAZ PAGODE | Tá Na Medida + Rubynho (ex Oz Bambaz),RECANTO SUCUPIRA,https://www.sympla.com.br/evento/santo-de-casa-faz-pagode-ta-na-me...
7,sympla,2026-08-22T19:00:00+00:00,Pagode 90º Convida Délcio Luiz - Brasília DF,Estação Beira Lago,https://www.sympla.com.br/evento/pagode-90o-convida-delcio-luiz-br...
8,sympla,2026-08-29T22:00:00+00:00,Arlindinho & Thiago Soares Pagode do Quadradinho,Local a definir,https://www.sympla.com.br/evento/arlindinho-thiago-soares-pagode-d...
9,ingresse,2026-12-05T18:00:00+00:00,Sorriso Eu Gosto no Pagode - Brasília,Parque Na Praia,https://www.ingresse.com/sorriso-eu-gosto-no-pagode-brasilia


In [4]:
# próximos eventos a partir de agora
from datetime import datetime, timezone
agora = datetime.now(timezone.utc).isoformat()
q("SELECT fonte, start_date, nome, local_nome FROM eventos WHERE start_date>=? ORDER BY start_date LIMIT 30", agora)

,fonte,start_date,nome,local_nome
0,sympla,2026-07-05T15:00:00+00:00,Varanda da Copa | Oitavas de Final + Pagode & Feijoada,Contexto Bar e Restaurante
1,sympla,2026-07-05T15:00:00+00:00,Copa no Lago - Clima de Montanha + Brasil nas Oitavas de Final,Estação Beira Lago
2,sympla,2026-07-05T15:00:00+00:00,Varanda da Copa | Brasil vs Noruega + Pagode + Feijoada | ADA no C...,Contexto Bar
3,sympla,2026-07-05T15:00:00+00:00,Copa no Lago :: Oitavas de Final :: CLIMA DE MONTANHA,Estação Beira Lago
4,sympla,2026-07-05T15:00:00+00:00,05.07 - TNP - Pagode do Jorgin,Bahrein Comida de Boteco
5,sympla,2026-07-05T16:00:00+00:00,Café com Hanz - Fanmeeting,BLOCO A
6,sympla,2026-07-05T16:30:00+00:00,BRASIL X NORUEGA COM BLOCO DA TIA ZÉLIA & ELAS QUE TOQUEM,Festival EM CASA BRASÍLIA
7,sympla,2026-07-05T17:00:00+00:00,ARENA SIMPRÃO | DOMINGO - JOGO BRASIL - CESINHA,Buteco do Simprão
8,sympla,2026-07-05T17:00:00+00:00,Copa do Mundo no Brazolia - Oitavas de Finais,Brazolia Cozinha E Bar
9,sympla,2026-07-05T17:00:00+00:00,Brasil + pagode,Vera cruz gastrobar


In [5]:
# célula livre — escreva seu próprio SQL, ex.:
q("SELECT * FROM eventos WHERE nome LIKE '%funk%'")

,id,fonte,id_nativo,nome,start_date,end_date,cidade,estado,local_nome,endereco,lat,lon,categoria,organizador,url,imagem,raspado_em
0,sympla:3486177,sympla,3486177,We The Funk convida Joãozinho VT | 09.07 | O Funk é Nosso!,2026-07-10T00:00:00+00:00,2026-07-10T07:00:00+00:00,Brasília,DF,Contexto Bar e Restaurante,SCES Trecho 2,-15.814470,-47.849760,NORMAL,Contexto Bar & Restaurante,https://www.sympla.com.br/evento/we-the-funk-convida-joaozinho-vt-...,https://images.sympla.com.br/6a495e4cb4811-lg.png,2026-07-05T12:48:04.144246+00:00
1,sympla:3461542,sympla,3461542,DOMINGO QUE GERA FUNK,2026-07-13T00:00:00+00:00,2026-07-13T08:00:00+00:00,Brasília,DF,INTENSE LOUNGES,Polo de desenvolvimento jucelino kubitschek trecho 01 conjunto 06 ...,-15.846416,-48.009109,NORMAL,Intense Lounges,https://www.sympla.com.br/evento/domingo-que-gera-funk/3461542,https://images.sympla.com.br/6a3cb4c288baa-lg.jpg,2026-07-05T12:48:04.144366+00:00
2,sympla:3461544,sympla,3461544,DOMNINGO QUE GERA FUNK,2026-07-20T00:00:00+00:00,2026-07-20T08:00:00+00:00,Brasília,DF,INTENSE LOUNGES,Polo de desenvolvimento jucelino kubitschek trecho 01 conjunto 06 ...,-15.846416,-48.009109,NORMAL,Intense Lounges,https://www.sympla.com.br/evento/domningo-que-gera-funk/3461544,https://images.sympla.com.br/6a3cb53308194-lg.jpg,2026-07-05T12:48:04.144403+00:00
3,sympla:3478109,sympla,3478109,FESTA REGALO ~ FUNK É GOSTOSO DEMAIS,2026-07-11T01:00:00+00:00,2026-07-11T08:00:00+00:00,Brasília,DF,EXTERNA,Centro Comercial Amazonas,-15.811417,-47.894501,NORMAL,Externa,https://www.sympla.com.br/evento/festa-regalo-funk-e-gostoso-demai...,https://images.sympla.com.br/6a43fa62aaf91-lg.jpg,2026-07-05T12:48:47.660560+00:00
4,sympla:3469018,sympla,3469018,Joga Ginga | Afrobeats - Reggaeton - Dancehall e Funk,2026-07-12T00:00:00+00:00,2026-07-12T08:00:00+00:00,Brasília,DF,Astcu,St. de Clubes Esportivos Sul Trecho 2,-15.816618,-47.845354,NORMAL,HC PRODUÇÕES,https://www.sympla.com.br/evento/joga-ginga-afrobeats-reggaeton-da...,https://images.sympla.com.br/6a38755cbfb31-lg.jpg,2026-07-05T12:48:47.660582+00:00
5,sympla:3461538,sympla,3461538,DOMINGO QUE GERA FUNK,2026-07-06T00:00:00+00:00,2026-07-06T09:00:00+00:00,Brasília,DF,INTENSE LOUNGES,Polo de desenvolvimento jucelino kubitschek trecho 01 conjunto 06 ...,-15.846416,-48.009109,NORMAL,Intense Lounges,https://www.sympla.com.br/evento/domingo-que-gera-funk/3461538,https://images.sympla.com.br/6a3cb17ab0943-lg.jpg,2026-07-05T12:48:47.660726+00:00
6,sympla:3429585,sympla,3429585,"Nexo: Funk, rap e house em Brasília",2026-11-28T00:00:00+00:00,2026-11-30T05:00:00+00:00,Brasília,DF,Local a definir,None,-15.797515,-47.891887,NORMAL,NEXO,https://www.sympla.com.br/evento/nexo-funk-rap-e-house-em-brasilia...,https://images.sympla.com.br/5f282a0c6a2d9.png,2026-07-05T12:48:47.660825+00:00
7,sympla:3461547,sympla,3461547,DOMINGO QUE GERA FUNK,2026-07-27T00:00:00+00:00,2026-07-27T08:00:00+00:00,Brasília,DF,INTENSE LOUNGES,Polo de desenvolvimento jucelino kubitschek trecho 01 conjunto 06 ...,-15.846416,-48.009109,NORMAL,Intense Lounges,https://www.sympla.com.br/evento/domingo-que-gera-funk/3461547,https://images.sympla.com.br/6a3cb5a43a619-lg.jpg,2026-07-05T12:48:47.660843+00:00
